# Telco Customer Churn Prediction - End-to-End ML Pipeline

This notebook builds, trains, and evaluates complete Scikit-Learn pipelines using custom transformers:
- **`clean_cls`**: Automated data cleaning, type conversion, binary encoding, and dummy variables.
- **`CorrelationThresholdFilter`**: Feature selection based on Pearson correlation with the target.
- **`DecisionTreeClassifier`** & **`LogisticRegression`**: Classification models.

In [39]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve

from cleaningcls import clean_cls
from ctf import CorrelationThresholdFilter

## 1. Load Data & Train-Test Split
We pass raw features into the pipeline; the custom cleaning transformer handles all data preprocessing automatically.

In [40]:
# Load raw dataset
df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')

# Separate features (X) and target (y)
X = df.drop(columns=['Churn'])
y = df['Churn'].map({'Yes': 1, 'No': 0})

# Stratified Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set shape: {X_train.shape}")
print(f"Test set shape: {X_test.shape}")
print("\nTarget distribution in train set:")
print(y_train.value_counts(normalize=True).round(3))

Training set shape: (5634, 20)
Test set shape: (1409, 20)

Target distribution in train set:
Churn
0    0.735
1    0.265
Name: proportion, dtype: float64


## 2. Decision Tree Pipeline
A pipeline containing `clean_cls`, `CorrelationThresholdFilter`, and `DecisionTreeClassifier`.

In [41]:
pipeline_dt = Pipeline([
    ('clean', clean_cls()),
    ('ctf', CorrelationThresholdFilter(threshold=0.23)),
    ('model', DecisionTreeClassifier(criterion='entropy',class_weight='balanced', max_depth=4, random_state=42))
])

# Fit pipeline on training data
pipeline_dt.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('clean', ...), ('ctf', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,threshold,0.23
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.",'entropy'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split.",'best'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",4
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0


In [42]:
# Predictions & Evaluation
y_prob_dt = pipeline_dt.predict_proba(X_test)[:, 1]
y_pred_dt = pipeline_dt.predict(X_test)
# y_pred_dt = (pipeline_dt.predict_proba(X_test)[: , 1] > 0.45).astype(int)

auc_dt = roc_auc_score(y_test, y_prob_dt)
print('=== Decision Tree Pipeline Evaluation ===')
print(f'ROC-AUC Score: {auc_dt:.4f}')
print('\nConfusion Matrix:')
print(confusion_matrix(y_test, y_pred_dt))
print('\nClassification Report:')
print(classification_report(y_test, y_pred_dt))


=== Decision Tree Pipeline Evaluation ===
ROC-AUC Score: 0.8219

Confusion Matrix:
[[781 254]
 [ 94 280]]

Classification Report:
              precision    recall  f1-score   support

           0       0.89      0.75      0.82      1035
           1       0.52      0.75      0.62       374

    accuracy                           0.75      1409
   macro avg       0.71      0.75      0.72      1409
weighted avg       0.79      0.75      0.76      1409



In [43]:
import pandas as pd

# 1. Get the trained model from your pipeline
model = pipeline_dt['model']

# 2. Get the feature importances (scores of how much the tree uses each feature)
importances = model.feature_importances_

# 3. Get the feature names from your pipeline's preprocessing steps

feature_names = pipeline_dt[:-1].get_feature_names_out()


# 4. Combine them into a clean DataFrame and sort by most important
feature_importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

# important_f = feature_importance_df[feature_importance_df['Importance'] > 0]
active_features = feature_importance_df

print("=== Features Used by the Decision Tree (Ranked by Importance) ===")
print(active_features.to_string(index=False))

=== Features Used by the Decision Tree (Ranked by Importance) ===
                       Feature  Importance
             Contract_Two year    0.454197
                        tenure    0.262647
   InternetService_Fiber optic    0.219740
PaymentMethod_Electronic check    0.063417


In [44]:
feature_names

Index(['tenure', 'InternetService_Fiber optic', 'Contract_Two year',
       'PaymentMethod_Electronic check'],
      dtype='object')

In [45]:

sample_output = pipeline_dt['clean'].transform(X_train.head(5))
sample_output = pipeline_dt['ctf'].transform(sample_output)
feature_names = list(sample_output.columns)

In [46]:
feature_names

['tenure',
 'InternetService_Fiber optic',
 'Contract_Two year',
 'PaymentMethod_Electronic check']

In [47]:
# 1. Transform a sample (e.g., the first row of your training data) through your pipeline steps
X_transformed = pipeline_dt['clean'].transform(X_train.head(1))
X_filtered = pipeline_dt['ctf'].transform(X_transformed)

# 2. Get the decision path for this specific sample from the model step
tree_model = pipeline_dt['model']
node_indicator = tree_model.decision_path(X_filtered)

# 3. Find which node IDs this sample passed through
leave_id = tree_model.apply(X_filtered)
node_index = node_indicator.indices[node_indicator.indptr[0]:node_indicator.indptr[1]]

print(f"Customer's Decision Path (Node IDs): {node_index}")
print(f"Final Leaf Node: {leave_id[0]}")

Customer's Decision Path (Node IDs): [0 1 2 6 7]
Final Leaf Node: 7


In [68]:
tree = pipeline_dt['model'].tree_
feature_names = pipeline_dt['ctf'].get_feature_names_out()
sample_customer = X_filtered.iloc[0] 

for node_id in node_index:
    if tree.value[7][0][1] <= 0.45:    
        if tree.children_left[node_id] != tree.children_right[node_id]:
            feat_idx = tree.feature[node_id]
            thresh = tree.threshold[node_id]
            feat_name = feature_names[feat_idx]
            
            # Get this specific customer's value for this feature
            customer_value = sample_customer[feat_name]
            
            print(f"Node {node_id} (Split Node):")
            print(f"  - Feature checked: '{feat_name}'")
            print(f"  - Tree rule: <= {thresh:.4f}")
            print(f"  - Customer's actual value: ",thresh <= customer_value)
        else:
            print(f"Node {node_id} (Leaf Node - Final Destination):")
            print(f"  - Class distribution (No Churn, Churn): {tree.value[node_id]}")
            print("=" * 40)
    # else:
    #     pass
    

Node 0 (Split Node):
  - Feature checked: 'Contract_Two year'
  - Tree rule: <= 0.5000
  - Customer's actual value:  False
Node 1 (Split Node):
  - Feature checked: 'InternetService_Fiber optic'
  - Tree rule: <= 0.5000
  - Customer's actual value:  False
Node 2 (Split Node):
  - Feature checked: 'tenure'
  - Tree rule: <= 5.5000
  - Customer's actual value:  True
Node 6 (Split Node):
  - Feature checked: 'PaymentMethod_Electronic check'
  - Tree rule: <= 0.5000
  - Customer's actual value:  False
Node 7 (Leaf Node - Final Destination):
  - Class distribution (No Churn, Churn): [[0.763877 0.236123]]
